# Sp500 index daily

link: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks?select=sp500_index.csv

In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("../data/01_raw/sp500_index.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df = df.set_index("Date").asfreq("D").ffill().reset_index()
df["item_id"] = "SP500"
df

,Date,S&P500,item_id
0,2014-12-22,2078.54,SP500
1,2014-12-23,2082.17,SP500
2,2014-12-24,2081.88,SP500
3,2014-12-25,2081.88,SP500
4,2014-12-26,2088.77,SP500
...,...,...,...
3647,2024-12-16,6074.08,SP500
3648,2024-12-17,6050.61,SP500
3649,2024-12-18,5872.16,SP500
3650,2024-12-19,5867.08,SP500


In [3]:
PREDICTION_LENGTH=30
TARGET="S&P500"

In [4]:
train_data = TimeSeriesDataFrame.from_data_frame(
    df, id_column="item_id", timestamp_column="Date"
)

train_split, test_split = train_data.train_test_split(
    prediction_length=PREDICTION_LENGTH
)

predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=[0.025, 0.05, 0.5, 0.95 ,0.975],
    target=TARGET,
    eval_metric="WAPE",
)

predictor.fit(
    train_split,
    hyperparameters={
        "RecursiveTabular": {
            "tabular_hyperparameters": {"GBM": {}, "XGB": {}, "CAT": {}}
        },
        "Naive": {},
        "SeasonalNaive": {},
        "PatchTST": {},
        "ETS": {},
        "ADIDA": {},
        "DLinear": {},
        "Toto": {},
        "ARIMA": {},
        "Theta": {},
        "DeepAR": {},
        "NPTS": {},
        "TemporalFusionTransformer": {},
        "Chronos2": {},
        "SimpleFeedForward": {},
        "TiDE":{},
        "WaveNet": {},
    },
)

Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/walmart_sales_benchmark_mlops/notebooks/AutogluonModels/ag-20260518_045449'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.62/11.62 GB
Total GPU Memory:   Free: 11.62 GB, Allocated: 0.00 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       18.32 GB / 31.15 GB (58.8%)
Disk Space Avail:   110.98 GB / 230.30 GB (48.2%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                     'DLinear': {},
                     'DeepAR': {},
                     '

In [5]:
leaderboard = predictor.leaderboard(train_split)

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


In [6]:
leaderboard

,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-0.010507,-0.010508,0.153851,0.108024,0.205914,18
1,PatchTST,-0.011177,-0.011177,0.016509,0.006436,21.515582,10
2,SimpleFeedForward,-0.011331,-0.011331,0.012872,0.005689,11.746289,16
3,DLinear,-0.011405,-0.011405,0.010011,0.006800,21.198540,14
4,TemporalFusionTransformer,-0.011550,-0.011550,0.036938,0.010091,32.625190,8
5,Chronos2,-0.012132,-0.012132,0.903795,0.276452,2.600151,7
6,TiDE,-0.012232,-0.012232,0.019816,0.007907,58.862487,11
7,RecursiveTabular,-0.013232,-0.013232,0.064847,0.061068,0.417245,3
8,DeepAR,-0.013290,-0.012896,0.076279,0.077463,49.993927,9
9,Toto,-0.013646,-0.013747,2.527668,0.989011,1.575875,17


In [7]:
predictions = predictor.predict(train_split, model="WeightedEnsemble")

In [8]:
selected_item = "SP500"

real_data = test_split.loc[selected_item]
predictions_store = predictions.loc[selected_item]

In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=real_data.index,
        y=real_data[TARGET],
        name="Actual S&P500 Price",
        mode="lines",
        line=dict(color="#1f77b4", width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["mean"],
        name="Forecast (Mean)",
        mode="lines",
        line=dict(color="#ff7f0e", width=2, dash="dash"),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.975"],
        name="Upper Bound (p97.5)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.025"],
        name="95% Confidence Interval (p2.5 - p97.5)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(255, 127, 14, 0.15)",
        line=dict(width=0),
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.95"],
        name="Upper Bound (p95)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        x=predictions_store.index,
        y=predictions_store["0.05"],
        name="90% Confidence Interval (p5 - p95)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(200, 100, 14, 0.15)",
        line=dict(width=0),
    )
)

fig.update_layout(
    title=f"Backtesting (Actual vs Forecast) - {selected_item}",
    xaxis_title="Date",
    yaxis_title="S&P 500 Index Price ($)",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

## Weekly Analysis & Forecasting

In this section, we group the daily data by week, using the value from the start of the week (Monday) and omitting the other days, in order to perform a weekly forecast analysis.

In [ ]:
df_weekly = df.set_index("Date").resample("W-MON").first().reset_index()

df_weekly["item_id"] = "SP500_weekly"
PREDICTION_LENGTH_WEEKLY = 13  
TARGET_WEEKLY = "S&P500"

df_weekly.head(10)

,Date,S&P500,item_id
0,2014-12-22,2078.54,SP500_weekly
1,2014-12-29,2082.17,SP500_weekly
2,2015-01-05,2080.35,SP500_weekly
3,2015-01-12,2002.61,SP500_weekly
4,2015-01-19,2023.03,SP500_weekly
5,2015-01-26,2022.55,SP500_weekly
6,2015-02-02,2029.55,SP500_weekly
7,2015-02-09,2050.03,SP500_weekly
8,2015-02-16,2068.59,SP500_weekly
9,2015-02-23,2100.34,SP500_weekly


In [ ]:
train_data_weekly = TimeSeriesDataFrame.from_data_frame(
    df_weekly, id_column="item_id", timestamp_column="Date"
)

train_split_weekly, test_split_weekly = train_data_weekly.train_test_split(
    prediction_length=PREDICTION_LENGTH_WEEKLY
)


predictor_weekly = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH_WEEKLY,
    quantile_levels=[0.025, 0.05, 0.5, 0.95, 0.975],
    target=TARGET_WEEKLY,
    eval_metric="WAPE",
)

predictor_weekly.fit(
    train_split_weekly,
    hyperparameters={
        "RecursiveTabular": {
            "tabular_hyperparameters": {"GBM": {}, "XGB": {}, "CAT": {}}
        },
        "Naive": {},
        "SeasonalNaive": {},
        "PatchTST": {},
        "ETS": {},
        "ADIDA": {},
        "DLinear": {},
        "Toto": {},
        "ARIMA": {},
        "Theta": {},
        "DeepAR": {},
        "NPTS": {},
        "TemporalFusionTransformer": {},
        "Chronos2": {},
        "SimpleFeedForward": {},
        "TiDE": {},
        "WaveNet": {},
    },
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260518_045855"
Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/walmart_sales_benchmark_mlops/notebooks/AutogluonModels/ag-20260518_045855'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 10.59/11.62 GB
Total GPU Memory:   Free: 10.59 GB, Allocated: 1.03 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       15.05 GB / 31.15 GB (48.3%)
Disk Space Avail:   110.98 GB / 230.30 GB (48.2%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
            

In [12]:
leaderboard_weekly = predictor_weekly.leaderboard(train_split_weekly)
leaderboard_weekly

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-0.011647,-0.011371,0.064861,0.043213,0.201319,18
1,DLinear,-0.013158,-0.013158,0.008940,0.005419,9.735653,14
2,SimpleFeedForward,-0.014130,-0.014130,0.010805,0.005115,16.588501,16
3,ETS,-0.014891,-0.014891,0.013550,0.013367,0.004457,4
4,TiDE,-0.015611,-0.015611,0.018499,0.007057,34.854047,11
5,WaveNet,-0.015678,-0.015186,0.028710,0.025471,10.966705,12
6,PatchTST,-0.015894,-0.015894,0.015362,0.005890,10.536118,10
7,ARIMA,-0.016109,-0.016109,0.013515,0.013267,0.004422,15
8,RecursiveTabular,-0.016171,-0.016171,0.039703,0.036394,0.286592,3
9,Theta,-0.016173,-0.016173,0.326066,0.013001,0.004322,6


In [13]:
predictions_weekly = predictor_weekly.predict(train_split_weekly, model="WeightedEnsemble")

In [14]:
selected_item_weekly = "SP500_weekly"

real_data_weekly = test_split_weekly.loc[selected_item_weekly]
predictions_weekly_store = predictions_weekly.loc[selected_item_weekly]

fig_weekly = go.Figure()

fig_weekly.add_trace(
    go.Scatter(
        x=real_data_weekly.index,
        y=real_data_weekly[TARGET_WEEKLY],
        name="Actual S&P500 Price (Weekly)",
        mode="lines",
        line=dict(color="#1f77b4", width=2),
    )
)

fig_weekly.add_trace(
    go.Scatter(
        x=predictions_weekly_store.index,
        y=predictions_weekly_store["mean"],
        name="Forecast (Mean)",
        mode="lines",
        line=dict(color="#ff7f0e", width=2, dash="dash"),
    )
)

fig_weekly.add_trace(
    go.Scatter(
        x=predictions_weekly_store.index,
        y=predictions_weekly_store["0.975"],
        name="Upper Bound (p97.5)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig_weekly.add_trace(
    go.Scatter(
        x=predictions_weekly_store.index,
        y=predictions_weekly_store["0.025"],
        name="95% Confidence Interval (p2.5 - p97.5)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(255, 127, 14, 0.15)",
        line=dict(width=0),
    )
)

fig_weekly.add_trace(
    go.Scatter(
        x=predictions_weekly_store.index,
        y=predictions_weekly_store["0.95"],
        name="Upper Bound (p95)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig_weekly.add_trace(
    go.Scatter(
        x=predictions_weekly_store.index,
        y=predictions_weekly_store["0.05"],
        name="90% Confidence Interval (p5 - p95)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(200, 100, 14, 0.15)",
        line=dict(width=0),
    )
)

fig_weekly.update_layout(
    title=f"Backtesting Semanal (Actual vs Forecast) - {selected_item_weekly}",
    xaxis_title="Date",
    yaxis_title="S&P 500 Index Price ($)",
    template="plotly_white",
    hovermode="x unified",
)
fig_weekly.show()

## Monthly Analysis & Forecasting

In this section, we resample the daily S&P 500 index data to a monthly frequency. We select the first available trading day's value of each month and skip the remaining days. We then train an AutoGluon TimeSeriesPredictor to forecast the index price at a monthly resolution.

In [15]:
df_monthly = df.set_index("Date").resample("MS").first().reset_index()

df_monthly["item_id"] = "SP500_monthly"
PREDICTION_LENGTH_MONTHLY = 12  
TARGET_MONTHLY = "S&P500"

df_monthly.head(12)

,Date,S&P500,item_id
0,2014-12-01,2078.54,SP500_monthly
1,2015-01-01,2058.90,SP500_monthly
2,2015-02-01,1994.99,SP500_monthly
3,2015-03-01,2104.50,SP500_monthly
4,2015-04-01,2059.69,SP500_monthly
5,2015-05-01,2108.29,SP500_monthly
6,2015-06-01,2111.73,SP500_monthly
7,2015-07-01,2077.42,SP500_monthly
8,2015-08-01,2103.84,SP500_monthly
9,2015-09-01,1913.85,SP500_monthly


In [16]:
train_data_monthly = TimeSeriesDataFrame.from_data_frame(
    df_monthly, id_column="item_id", timestamp_column="Date"
)

train_split_monthly, test_split_monthly = train_data_monthly.train_test_split(
    prediction_length=PREDICTION_LENGTH_MONTHLY
)

predictor_monthly = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH_MONTHLY,
    quantile_levels=[0.025, 0.05, 0.5, 0.95, 0.975],
    target=TARGET_MONTHLY,
    eval_metric="WAPE",
)

predictor_monthly.fit(
    train_split_monthly,
    hyperparameters={
        "RecursiveTabular": {
            "tabular_hyperparameters": {"GBM": {}, "XGB": {}, "CAT": {}}
        },
        "Naive": {},
        "SeasonalNaive": {},
        "PatchTST": {},
        "ETS": {},
        "ADIDA": {},
        "DLinear": {},
        "Toto": {},
        "ARIMA": {},
        "Theta": {},
        "DeepAR": {},
        "NPTS": {},
        "TemporalFusionTransformer": {},
        "Chronos2": {},
        "SimpleFeedForward": {},
        "TiDE": {},
        "WaveNet": {},
    },
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260518_050120"
Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/walmart_sales_benchmark_mlops/notebooks/AutogluonModels/ag-20260518_050120'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.61/11.62 GB
Total GPU Memory:   Free: 11.61 GB, Allocated: 0.01 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       14.26 GB / 31.15 GB (45.8%)
Disk Space Avail:   110.98 GB / 230.30 GB (48.2%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
            

In [17]:
leaderboard_monthly = predictor_monthly.leaderboard(train_split_monthly)
leaderboard_monthly

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-0.032218,-0.032218,1.226604,0.422296,0.208043,18
1,Theta,-0.047517,-0.047517,0.296205,0.013069,0.004675,6
2,Naive,-0.056340,-0.056340,0.012984,0.013689,0.005125,1
3,DeepAR,-0.060062,-0.053264,0.035581,0.028899,7.991600,9
4,ARIMA,-0.063919,-0.063919,0.013325,0.013285,0.004414,15
5,ETS,-0.064382,-0.064382,0.043576,0.043419,0.005057,4
6,Toto,-0.067146,-0.067200,1.724977,0.120504,1.584159,17
7,Chronos2,-0.069169,-0.069169,0.905747,0.016184,0.918376,7
8,SimpleFeedForward,-0.071509,-0.071509,0.010393,0.005681,7.780708,16
9,ADIDA,-0.074081,-0.074081,0.013161,0.385605,0.004191,13


In [ ]:
predictions_monthly = predictor_monthly.predict(train_split_monthly, model="WeightedEnsemble")

In [23]:
selected_item_monthly = "SP500_monthly"

real_data_monthly = test_split_monthly.loc[selected_item_monthly]
predictions_monthly_store = predictions_monthly.loc[selected_item_monthly]

fig_monthly = go.Figure()

fig_monthly.add_trace(
    go.Scatter(
        x=real_data_monthly.index,
        y=real_data_monthly[TARGET_MONTHLY],
        name="Actual S&P500 Price (Monthly)",
        mode="lines",
        line=dict(color="#1f77b4", width=2),
    )
)


fig_monthly.add_trace(
    go.Scatter(
        x=predictions_monthly_store.index,
        y=predictions_monthly_store["mean"],
        name="Forecast (Mean)",
        mode="lines",
        line=dict(color="#ff7f0e", width=2, dash="dash"),
    )
)


fig_monthly.add_trace(
    go.Scatter(
        x=predictions_monthly_store.index,
        y=predictions_monthly_store["0.975"],
        name="Upper Bound (p97.5)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig_monthly.add_trace(
    go.Scatter(
        x=predictions_monthly_store.index,
        y=predictions_monthly_store["0.025"],
        name="95% Confidence Interval (p2.5 - p97.5)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(255, 127, 14, 0.15)",
        line=dict(width=0),
    )
)

fig_monthly.add_trace(
    go.Scatter(
        x=predictions_monthly_store.index,
        y=predictions_monthly_store["0.95"],
        name="Upper Bound (p95)",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)

fig_monthly.add_trace(
    go.Scatter(
        x=predictions_monthly_store.index,
        y=predictions_monthly_store["0.05"],
        name="90% Confidence Interval (p5 - p95)",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(200, 100, 14, 0.15)",
        line=dict(width=0),
    )
)

fig_monthly.update_layout(
    title=f"Monthly Backtesting (Actual vs Forecast) - {selected_item_monthly}",
    xaxis_title="Date",
    yaxis_title="S&P 500 Index Price ($)",
    template="plotly_white",
    hovermode="x unified",
)
fig_monthly.show()